# MicrobiomeDataSpace and MAGraph



In [ ]:
! sh ./Metabiome/mdo_workshop_setup.sh  > /dev/null 2>&1 # take < 3 min for run
# then select the new kernel named metabiome

[+] 0.0s
[+] 0.1s
conda-forge/linux-aarch64 ━━━━━━━╸━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.0s
conda-forge/noarch        ╸━━━━━━━━━━━━━━━╸━   0.0 B /  ??.?MB @  ??.?MB/s  0.0s[+] 0.2s
conda-forge/linux-aarch64 ━━━━━━━━━━━━━━━━━━ 262.1kB /  19.3MB @   1.4MB/s  0.1s
conda-forge/noarch        ━━━━━━━━━━━━━━━━━━ 785.0kB /  22.6MB @   4.1MB/s  0.1s[+] 0.3s
conda-forge/linux-aarch64 ╸━━━━━━━━━━━━━━━━━   2.3MB /  19.3MB @   7.6MB/s  0.2s
conda-forge/noarch        ━╸━━━━━━━━━━━━━━━━   3.2MB /  22.6MB @  10.7MB/s  0.2s[+] 0.4s
conda-forge/linux-aarch64 ━━━╸━━━━━━━━━━━━━━   4.8MB /  19.3MB @  13.7MB/s  0.3s
conda-forge/noarch        ━━╸━━━━━━━━━━━━━━━   4.1MB /  22.6MB @  11.5MB/s  0.3s[+] 0.5s
conda-forge/linux-aarch64 ━━━━━━━━╸━━━━━━━━━   9.9MB /  19.3MB @  21.8MB/s  0.4s
conda-forge/noarch        ━━━━━━╸━━━━━━━━━━━   9.7MB /  22.6MB @  21.2MB/s  0.4s[+] 0.6s
conda-forge/linux-aarch64 ━━━━━━━━━━━━━━╸━━━  17.0MB /  19.3MB @  30.6MB/s  0.5s
conda-forge/noarch        ━━━━━━━━━━╸━━━━━━━  14.7M

## An overview

*   

## Setup of the environment



In [1]:
import polars as pl
import metabiome.io as mio

## Load data

In [2]:
data_dir = "/biodata/resources/day3_lab5/nar_operon_query_output"
# Load from multiple file formats
mds = mio.from_files(
    obs="{}/input/metadata".format(data_dir),           # Sample metadata
    abundance="{}/input/RPKM.json".format(data_dir),   # Gene abundance profiles
    taxonomy="{}/input/taxonomy.json".format(data_dir), # Taxonomic annotations
    functional="{}/input/FG.json".format(data_dir),    # Functional groups
    sequences="{}/input/merged.fasta".format(data_dir), # Gene sequences
    id_mapping="{}/input/id_mapping.tsv".format(data_dir) # ID cross-references
)


In [6]:
# Check data dimensions
print(f"Data shape: {mds.shape}")  # (n_samples, n_features)

Data shape: (9522, 9484)


## Profile the abundance per species per sample


In [7]:
# aggregrate by pfam_domain and taxa first
operon_abd_data = mds.groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_diseaseMean = operon_abd_data.groupby.obs("disease_group").agg("mean")


In [ ]:
cur_species = "Veillonella parvula" #"Veillonella parvula", "Escherichia coli"
operon_abd_data.pl.boxplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col=None,
    title="Abundance of {0} by Disease Group".format(cur_species),
)

## Stratify by disease group

In [8]:
# operon_abd_data_CD = operon_abd_data.filter.obs(pl.col("disease_group") == "CD")
operon_abd_data_CD = mds.filter.obs(pl.col("disease_group") == "CD").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_Healthy = mds.filter.obs(pl.col("disease_group") == "Healthy").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")


/workspaces/course-pilot/Metabiome/src/metabiome/core/space.py:369: UserWarning: 1 filtered samples were not present in the matrix and were dropped
  warnings.warn(


In [ ]:
operon_abd_data_Healthy.pl.sankey(
    hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.1
)


In [15]:
operon_abd_data_CD = mds.filter.obs(pl.col("disease_group") == "CD").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_CD.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.1
)

### Task: 

